# 02 - Dataset Profile and EDA

Learn how to audit geospatial datasets before any modeling.


## Definition
Explain the concept in one sentence and why it matters in delivery analytics.

## Theory
Describe the assumptions and algorithmic behavior at a practical level.

## Mathematical Intuition
Show the formula or geometry intuition behind the method.

## Visual Explanation
Use a chart/map to make the concept concrete.

## Code Explanation
Walk through what each code block does and what inputs/outputs mean.

## Interpretation of Results
Connect metrics and visuals to operational decisions.


In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

if (Path.cwd() / "src").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise RuntimeError("Could not locate project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Geospatial Clustering


In [2]:
from src.config import TRAIN_FILE_PATH
from src.data_loader import load_raw_data, load_and_clean_data, explain_dataset_fields


In [3]:
from src.data_quality import run_quality_gate, quality_gate_to_markdown

raw_df = load_raw_data(TRAIN_FILE_PATH)
quality = run_quality_gate(raw_df)
print(quality_gate_to_markdown(quality))

fields = explain_dataset_fields()
display(pd.DataFrame({"field": fields.keys(), "explanation": fields.values()}))


## Dataset Quality Gate
- Rows: 45,593
- Columns: 20
- Duplicate rows: 0
- Duplicate IDs: 0
- Global-bound violations: 0
- Out-of-India rows: 4,071
- Zero-coordinate rows: 3,640
- Potential coordinate swaps: 0
- Distance p90 (km): 19.40
- Warnings:
  - 4071 rows are outside India bounds and should be cleaned.
  - 3640 rows include zero coordinates and may be GPS errors.
  - Extreme delivery distances detected (>500 km); likely geocoding or data-entry anomalies.


,field,explanation
0,ID,Unique order identifier.
1,Delivery_person_ID,Unique delivery partner identifier.
2,Delivery_person_Age,Age of delivery partner in years.
3,Delivery_person_Ratings,Historical rating score of delivery partner.
4,Restaurant_latitude,Latitude of restaurant pickup location.
5,Restaurant_longitude,Longitude of restaurant pickup location.
6,Delivery_location_latitude,Latitude of customer drop location.
7,Delivery_location_longitude,Longitude of customer drop location.
8,Order_Date,Date when the order was placed.
9,Time_Orderd,Clock time when order was placed.


In [4]:
clean_df = load_and_clean_data(TRAIN_FILE_PATH)
print("Raw shape:", raw_df.shape)
print("Clean shape:", clean_df.shape)
display(clean_df.head())


Raw shape: (45593, 20)
Clean shape: (40007, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,2,Snack,motorcycle,0.0,No,Urban,24
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,2,Snack,scooter,1.0,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,0,Drinks,motorcycle,1.0,No,Urban,26
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,0,Buffet,motorcycle,1.0,No,Metropolitian,21
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,1,Snack,scooter,1.0,No,Metropolitian,30


In [5]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
clean_df["Restaurant_latitude"].hist(ax=axes[0,0], bins=50)
axes[0,0].set_title("Restaurant Latitude")

clean_df["Restaurant_longitude"].hist(ax=axes[0,1], bins=50)
axes[0,1].set_title("Restaurant Longitude")

clean_df["Delivery_location_latitude"].hist(ax=axes[1,0], bins=50)
axes[1,0].set_title("Delivery Latitude")

clean_df["Delivery_location_longitude"].hist(ax=axes[1,1], bins=50)
axes[1,1].set_title("Delivery Longitude")

plt.tight_layout()
plt.show()


**Interpretation**
- High anomaly rates in coordinates directly damage cluster quality.
- Data cleaning is not optional in geospatial pipelines.
